In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import librosa
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Dataset indexing**

**Label mapping**

In [2]:
ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
STEMS_PATH = os.path.join(ROOT, "genres_stems")

GENRES = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]

label_map = {g:i for i,g in enumerate(GENRES)}

rows = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = sorted(os.listdir(genre_dir))
    
    for s in songs:
        song_dir = os.path.join(genre_dir, s)
        if not os.path.isdir(song_dir):
            continue
        
        rows.append({
            "path": song_dir,
            "genre": g,
            "label": label_map[g]
        })

df = pd.DataFrame(rows)

print("Total samples:", len(df))
df.head()

Total samples: 1000


,path,genre,label
0,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues,0
1,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues,0
2,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues,0
3,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues,0
4,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues,0


**Tr/Val Split**

In [3]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))

Train samples: 800
Validation samples: 200


**Mel Spectrogram function**

In [4]:
def audio_to_melspec(path, sr=22050, duration=10, n_mels=128):
    
    stems = []
    for stem in ["drums.wav","vocals.wav","bass.wav","other.wav"]:
        file = os.path.join(path, stem)
        y, sr = librosa.load(file, sr=sr, duration=duration)
        stems.append(y)
    
    # make equal length
    min_len = min(len(s) for s in stems)
    stems = [s[:min_len] for s in stems]
    
    # mix stems
    mix = np.sum(stems, axis=0)
    
    # convert to mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=mix,
        sr=sr,
        n_mels=n_mels,
        fmax=sr//2
    )
    
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    return mel_db

In [5]:
# def audio_to_melspec(path, sr=22050, duration=10, n_mels=128):
#     # load only 'other.wav' stem for now (fast baseline)
#     file = os.path.join(path, "other.wav")
    
#     y, sr = librosa.load(file, sr=sr, duration=duration)
    
#     mel = librosa.feature.melspectrogram(
#         y=y,
#         sr=sr,
#         n_mels=n_mels,
#         fmax=sr//2
#     )
    
#     mel_db = librosa.power_to_db(mel, ref=np.max)
    
#     return mel_db

**PyTorch Dataset class**

In [6]:
class GenreDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel = audio_to_melspec(row["path"])
        
        mel = torch.tensor(mel).unsqueeze(0).float()
        label = torch.tensor(row["label"]).long()
        
        return mel, label

**Create Dataloaders**

In [7]:
train_dataset = GenreDataset(train_df)
val_dataset   = GenreDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 50
Validation batches: 13


**First CNN Architecture**

*Model from Scratch*

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.adapt = nn.AdaptiveAvgPool2d((8,8))
        
        self.fc1 = nn.Linear(32*8*8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = self.adapt(x)
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

model = SimpleCNN()
print(model)

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (adapt): AdaptiveAvgPool2d(output_size=(8, 8))
  (fc1): Linear(in_features=2048, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


**Training setup**

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Using device:", device)

Using device: cuda


**Training loop (first version)**

In [10]:
from tqdm import tqdm

def train_one_epoch(loader):
    model.train()
    total_loss = 0
    
    for x, y in tqdm(loader):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)


def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            
            correct += (pred == y).sum().item()
            total += y.size(0)
    
    return correct / total

**Training the CNN**

In [11]:
# epochs = 5

# for epoch in range(epochs):
#     train_loss = train_one_epoch(train_loader)
#     val_acc = evaluate(val_loader)
    
#     print(f"Epoch {epoch+1}/{epochs}")
#     print(f"Train Loss: {train_loss:.4f} | Val Accuracy: {val_acc:.4f}\n")

100%|██████████| 50/50 [02:46<00:00,  3.33s/it]


Epoch 1/5
Train Loss: 2.5336 | Val Accuracy: 0.2050



100%|██████████| 50/50 [01:32<00:00,  1.86s/it]


Epoch 2/5
Train Loss: 2.1636 | Val Accuracy: 0.1250



100%|██████████| 50/50 [01:32<00:00,  1.85s/it]


Epoch 3/5
Train Loss: 1.9730 | Val Accuracy: 0.3650



100%|██████████| 50/50 [01:31<00:00,  1.84s/it]


Epoch 4/5
Train Loss: 1.6454 | Val Accuracy: 0.4700



100%|██████████| 50/50 [01:31<00:00,  1.84s/it]


Epoch 5/5
Train Loss: 1.5075 | Val Accuracy: 0.4950



### Load test data

In [12]:
TEST_CSV = os.path.join(ROOT, "test.csv")
TEST_AUDIO = os.path.join(ROOT, "mashups")

test_df = pd.read_csv(TEST_CSV)
test_df.head()

,id,filename
0,1,mashups/song0001.wav
1,2,mashups/song0002.wav
2,3,mashups/song0003.wav
3,4,mashups/song0004.wav
4,5,mashups/song0005.wav


**Prediction function**

In [13]:
def predict_genre(file_path):
    y, sr = librosa.load(file_path, sr=22050, duration=10)
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel = librosa.power_to_db(mel, ref=np.max)
    
    mel = torch.tensor(mel).unsqueeze(0).unsqueeze(0).float().to(device)
    
    model.eval()
    with torch.no_grad():
        out = model(mel)
        pred = out.argmax(dim=1).item()
    
    return pred

**Generate predictions**

In [14]:
preds = []

for idx in tqdm(test_df["id"]):
    fname = f"song{str(idx).zfill(4)}.wav"
    file_path = os.path.join(TEST_AUDIO, fname)
    preds.append(predict_genre(file_path))

test_df["genre"] = [GENRES[p] for p in preds]
test_df.head()

100%|██████████| 3020/3020 [01:42<00:00, 29.44it/s]


,id,filename,genre
0,1,mashups/song0001.wav,reggae
1,2,mashups/song0002.wav,jazz
2,3,mashups/song0003.wav,pop
3,4,mashups/song0004.wav,blues
4,5,mashups/song0005.wav,country


**submission file**

In [15]:
submission = test_df[["id","genre"]]
submission.to_csv("submission.csv", index=False)
submission.head()

,id,genre
0,1,reggae
1,2,jazz
2,3,pop
3,4,blues
4,5,country
